In [1]:
import json
from collections import OrderedDict
from datetime import datetime
import re

import os
import sys

import yaml


In [2]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_path = os.path.join(config["paths"]["proj_store"], "data")
models_folderpath = config["paths"]["models"]


### IKAT

In [3]:

input_folder = f"{data_path}/raw_data/comps/information_seeking/ikat"
output_file = f"{data_path}/comps/ikat_dialogues.json"

all_dialogues = []

for filename in os.listdir(input_folder):
    if filename.endswith('.json'):
        file_path = os.path.join(input_folder, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            source_name = os.path.splitext(filename)[0].replace('_', '-')  # remove .json and replace underscores

            for dialogue in data:
                ordered_dialogue = OrderedDict()

                number = dialogue.get('number', 'unknown')
                ordered_dialogue['dialogue_id'] = f"{source_name}-{number}"

                for key, value in dialogue.items():
                    if key in ['ptkb', 'number']:
                        continue  # skip ptkb and number
                    elif key == 'turns':
                        split_turns = []
                        for turn in value:
                            turn.pop('ptkb_provenance', None)
                            turn.pop('response_provenance', None)

                            base_id = turn['turn_id']

                            # User turn
                            user_turn = {
                                'turn_id': f"{base_id}-1",
                                'role': 'user',
                                'utterance': turn['utterance']
                            }
                            split_turns.append(user_turn)

                            # System turn
                            system_turn = {
                                'turn_id': f"{base_id}-2",
                                'role': 'system',
                                'utterance': turn['response']
                            }
                            split_turns.append(system_turn)

                        ordered_dialogue['turns'] = split_turns
                    else:
                        ordered_dialogue[key] = value

                all_dialogues.append(ordered_dialogue)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_dialogues, f, indent=2, ensure_ascii=False)

### CAST

In [4]:

input_folder = f"{data_path}/raw_data/comps/information_seeking/cast"
output_file = f"{data_path}/comps/cast_dialogues.json"

all_conversations = []

for filename in os.listdir(input_folder):
    if filename.endswith('.json'):
        file_path = os.path.join(input_folder, filename)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            source_name = os.path.splitext(filename)[0].replace('_', '-')  # remove .json, replace underscores

            for convo in data:
                ordered_convo = OrderedDict()

                number = convo.get('number', 'unknown')
                ordered_convo['dialogue_id'] = f"{source_name}-{number}"

                for key, value in convo.items():
                    if key in ['number']:
                        continue  # skip number (already in dialogue_id)
                    elif key == 'turn':
                        split_turns = []
                        for turn in value:
                            turn.pop('provenance', None)

                            base_number = turn['number']
                            user_utt = turn.get('utterance') or turn.get('raw_utterance')
                            system_output = turn.get('response') or turn.get('passage')

                            if not user_utt or not system_output:
                                continue  # skip incomplete turns

                            # User turn
                            user_turn = {
                                'turn_id': f"{base_number}-1",
                                'role': 'user',
                                'utterance': user_utt
                            }
                            split_turns.append(user_turn)

                            # System turn
                            system_turn = {
                                'turn_id': f"{base_number}-2",
                                'role': 'system',
                                'utterance': system_output
                            }
                            split_turns.append(system_turn)

                        ordered_convo['turns'] = split_turns
                    else:
                        ordered_convo[key] = value

                all_conversations.append(ordered_convo)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_conversations, f, indent=2, ensure_ascii=False)

### DSTC8

In [5]:

input_folder = f"{data_path}/raw_data/comps/task_oriented/dstc8"
output_file = f"{data_path}/comps/dstc8_dialogues.json"

all_dialogues = []

for root, _, files in os.walk(input_folder):
    for filename in files:
        if filename.endswith('.json'):
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, input_folder)

            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for dialogue in data:
                    ordered_dialogue = OrderedDict()
                    ordered_dialogue['source_file'] = relative_path.replace("\\", "/")
                    ordered_dialogue['dialogue_id'] = dialogue['dialogue_id']

                    cleaned_turns = []
                    for turn in dialogue.get('turns', []):
                        cleaned_turn = {
                            'role': turn['speaker'].lower(),
                            'utterance': turn['utterance']
                        }
                        cleaned_turns.append(cleaned_turn)

                    ordered_dialogue['turns'] = cleaned_turns
                    all_dialogues.append(ordered_dialogue)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_dialogues, f, indent=2, ensure_ascii=False)


### MultiWOZ

In [6]:

input_folder = f"{data_path}/raw_data/comps/task_oriented/multiwoz"
output_file = f"{data_path}/comps/multiwoz_dialogues.json"

all_dialogues = []

for root, _, files in os.walk(input_folder):
    for filename in files:
        if filename.endswith('.json'):
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, input_folder)

            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for dialogue in data:
                    ordered_dialogue = OrderedDict()
                    ordered_dialogue['source_file'] = relative_path.replace("\\", "/")
                    ordered_dialogue['dialogue_id'] = dialogue['dialogue_id']

                    cleaned_turns = []
                    for turn in dialogue.get('turns', []):
                        cleaned_turn = {
                            'role': turn['speaker'].lower(),
                            'utterance': turn['utterance']
                        }
                        cleaned_turns.append(cleaned_turn)

                    ordered_dialogue['turns'] = cleaned_turns
                    all_dialogues.append(ordered_dialogue)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_dialogues, f, indent=2, ensure_ascii=False)



### BYU PCCL

In [7]:

input_folder = f"{data_path}/raw_data/comps/chit_chat/byu_pccl"
output_file = f"{data_path}/comps/byu_pccl_dialogues.json"


all_dialogues = []

for root, _, files in os.walk(input_folder):
    for filename in files:
        if filename.endswith('.json'):
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, input_folder)

            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

                for dialogue_id, dialogue_data in data.items():
                    ordered_dialogue = OrderedDict()
                    ordered_dialogue['source_file'] = relative_path.replace("\\", "/")
                    ordered_dialogue['dialogue_id'] = dialogue_id

                    participant_map = {}
                    participant_counter = 1

                    cleaned_turns = []
                    last_sender = None
                    last_turn = None

                    for message_group in dialogue_data.get('messages', []):
                        for message in message_group:
                            sender_id = message['sender']
                            if sender_id not in participant_map:
                                participant_map[sender_id] = f"participant_{participant_counter}"
                                participant_counter += 1

                            current_role = participant_map[sender_id]
                            current_text = message['text']
                            current_timestamp = message['timestamp']

                            if last_sender == sender_id:
                                # Append to last turn's utterance
                                last_turn['utterance'] += "\n\n" + current_text
                                # Keep the earliest timestamp
                                if current_timestamp < last_turn['timestamp']:
                                    last_turn['timestamp'] = current_timestamp
                            else:
                                # Start a new turn
                                new_turn = {
                                    'role': current_role,
                                    'utterance': current_text,
                                    'timestamp': current_timestamp
                                }
                                cleaned_turns.append(new_turn)
                                last_turn = new_turn
                                last_sender = sender_id

                    ordered_dialogue['turns'] = cleaned_turns
                    all_dialogues.append(ordered_dialogue)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_dialogues, f, indent=2, ensure_ascii=False)

print(f"Combined JSON saved to {output_file}")





Combined JSON saved to /data/yield-v1/data/comps/byu_pccl_dialogues.json


### Daily Dialogue

In [8]:

input_folder = f"{data_path}/raw_data/comps/chit_chat/dailydialog"
output_file = f"{data_path}/comps/dailydialogue_dialogues.json"




def clean_punctuation(text):
    # Fix spaced punctuation inside sentence
    text = re.sub(r'\s,\s', ', ', text)
    text = re.sub(r'\s\.\s', '. ', text)
    text = re.sub(r'\s\?\s', '? ', text)
    text = re.sub(r'\s!\s', '! ', text)
    text = re.sub(r"\s'\s", "'", text)
    text = re.sub(r"\s’\s", "’", text)
    # Remove space before punctuation at the end (final comma, period, question mark, exclamation)
    text = re.sub(r'\s([,.!?])$', r'\1', text)
    return text

all_dialogues = []

for root, _, files in os.walk(input_folder):
    for filename in files:
        if filename.endswith('.txt'):
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, input_folder)

            with open(file_path, 'r', encoding='utf-8') as f:
                for idx, line in enumerate(f):
                    line = line.strip()
                    if not line:
                        continue

                    utterances = [clean_punctuation(u.strip()) for u in line.split('__eou__') if u.strip()]
                    ordered_dialogue = OrderedDict()
                    ordered_dialogue['source_file'] = relative_path.replace("\\", "/")
                    ordered_dialogue['dialogue_id'] = f"{filename}_line{idx + 1}"

                    cleaned_turns = []
                    participant_toggle = 1

                    for utterance in utterances:
                        cleaned_turn = {
                            'role': f"participant_{participant_toggle}",
                            'utterance': utterance
                        }
                        cleaned_turns.append(cleaned_turn)
                        participant_toggle = 2 if participant_toggle == 1 else 1

                    ordered_dialogue['turns'] = cleaned_turns
                    all_dialogues.append(ordered_dialogue)

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(all_dialogues, f, indent=2, ensure_ascii=False)
